# Dota 2: baseline для предсказания исхода матча

В этом ноутбуке я собираю базовый ML-пайплайн для соревнования по Dota 2: от первичного анализа данных до sparse-признаков по драфту героев и подбора логистической регрессии. Формат оставлен как исследовательский отчёт: что проверялось, зачем и какой вывод получился.


## 1. Метрика и загрузка данных

Сначала фиксирую метрику соревнования и смотрю на базовую структуру train/test. Важно сразу понять баланс таргета, временное разбиение и пропуски, потому что это влияет на выбор validation protocol.


In [ ]:
from sklearn.metrics import roc_auc_score

def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1.0


In [ ]:
import pandas as pd

train_df = pd.read_csv("dota-2-hse-ml-1-course-competition-2026/matches_df_train.csv")
test_df = pd.read_csv("dota-2-hse-ml-1-course-competition-2026/matches_df_test.csv")


In [ ]:
print("matches_df_train")
print(train_df.shape)
display(train_df.sample(10))
display(train_df.describe(include="all"))
display(train_df.isna().sum())

print("matches_df_test")
print(test_df.shape)
display(test_df.sample(10))
display(test_df.describe(include="all"))
display(test_df.isna().sum())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(data=train_df, x="radiant_win")
plt.title("Распределение radiant_win")
plt.xlabel("radiant_win")
plt.ylabel("count")
plt.show()


**Вывод.** Целевая переменная достаточно сбалансирована, поэтому Gini/ROC-AUC подходит для сравнения моделей. Train содержит матчи до декабря 2024 года, test соответствует следующему периоду, поэтому случайный split может давать завышенную оценку.


### Наблюдения по данным

| Объект | Размер |
|---|---:|
| Train matches | 641,090 |
| Test matches | 59,748 |
| Target | `radiant_win` |

Train и test разделены по времени: test соответствует декабрю 2024 года. Поэтому основной риск — получить красивую локальную метрику на random split, которая не переносится на leaderboard.


## 2. Регион как категориальный признак

Проверяю, различается ли вероятность победы Radiant по регионам и можно ли извлечь из этого стабильный сигнал. Для кодирования использую target encoding, но дальше учитываю риск leakage при валидации.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.countplot(data=train_df, x="region", ax=axes[0, 0])
axes[0, 0].set_title("Train: регионы (абсолютное)")
axes[0, 0].tick_params(axis="x", labelrotation=40)

train_pct = train_df["region"].value_counts(normalize=True).sort_index() * 100
sns.barplot(x=train_pct.index, y=train_pct.values, ax=axes[0, 1])
axes[0, 1].set_title("Train: регионы (процентное)")
axes[0, 1].set_ylabel("percent")
axes[0, 1].tick_params(axis="x", labelrotation=40)

sns.countplot(data=test_df, x="region", ax=axes[1, 0])
axes[1, 0].set_title("Test: регионы (абсолютное)")
axes[1, 0].tick_params(axis="x", labelrotation=40)

test_pct = test_df["region"].value_counts(normalize=True).sort_index() * 100
sns.barplot(x=test_pct.index, y=test_pct.values, ax=axes[1, 1])
axes[1, 1].set_title("Test: регионы (процентное)")
axes[1, 1].set_ylabel("percent")
axes[1, 1].tick_params(axis="x", labelrotation=40)

plt.tight_layout()
plt.show()


In [ ]:
target_by_region = train_df.groupby("region")["radiant_win"].mean().sort_values(ascending=False)
display(target_by_region)
sns.barplot(target_by_region)
plt.xticks(rotation=40)
plt.ylabel("mean radiant_win")
plt.title("Среднее значение таргета по регионам (train)")
plt.tight_layout()
plt.show()


In [ ]:
from category_encoders import TargetEncoder

X_train = train_df.drop(columns=["radiant_win"]).copy()
y_train = train_df["radiant_win"].astype(int)

X_test = test_df.copy()

enc = TargetEncoder(cols=["region"])
X_train["region_te"] = enc.fit_transform(X_train[["region"]], y_train)["region"]
X_test["region_te"] = enc.transform(X_test[["region"]])["region"]


**Вывод.** Распределения регионов различаются, а средний `radiant_win` по регионам не одинаковый. Поэтому target encoding региона полезен, но требует fold-wise применения.


## 3. Даты и out-of-time validation

Test находится позже train по времени, поэтому обычный random split может быть слишком оптимистичным. Здесь я проверяю date features отдельно и сравниваю их с более честной time-aware валидацией.


In [ ]:
tmp = train_df.copy()
tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce")

win_rate_daily = tmp.groupby(tmp["date"].dt.date, as_index=False)["radiant_win"].mean()

sns.lineplot(data=win_rate_daily, x="date", y="radiant_win")
plt.xticks(rotation=40)
plt.ylabel("win rate")
plt.title("Доля побед Radiant по датам")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

for df in [X_train, X_test]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    df["is_weekend"] = (df["dayofweek"] > 4).astype(int)  # 5,6 - сб и вс

    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)  # делим на кол-во дней в неделе
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)  # тут 12 месяцев в году
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

order = X_train.sort_values("date").index
X_train = X_train.loc[order].reset_index(drop=True)
y_train = y_train.loc[order].reset_index(drop=True)

ts = TimeSeriesSplit(n_splits=5)

for tr_ind, val_ind in ts.split(X_train):
    assert X_train.iloc[tr_ind]["date"].max() <= X_train.iloc[val_ind]["date"].min()


In [ ]:
from sklearn.linear_model import LogisticRegression


def lr_gini(use_dates, use_region, ts):
    scores = []

    for tr_ind, val_ind in ts.split(X_train):
        X_tr = X_train.iloc[tr_ind].copy()
        X_val = X_train.iloc[val_ind].copy()
        y_tr = y_train.iloc[tr_ind]
        y_val = y_train.iloc[val_ind]

        cols = []
        if use_dates:
            cols += [
                "day",
                "is_weekend",
                "dow_sin",
                "dow_cos",
                "month_sin",
                "month_cos",
            ]
        if use_region:
            enc = TargetEncoder(cols=["region"])
            X_tr["region_te"] = enc.fit_transform(X_tr[["region"]], y_tr)["region"]
            X_val["region_te"] = enc.transform(X_val[["region"]])["region"]
            cols += ["region_te"]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_tr[cols], y_tr)
        p_val = model.predict_proba(X_val[cols])[:, 1]
        scores.append(gini(y_val, p_val))

    return np.mean(scores), np.std(scores)


for type, use_dates, use_region in [
    ("dates", True, False),
    ("regions", False, True),
    ("dates+regions", True, True),
]:
    m, s = lr_gini(use_dates, use_region, ts)
    print(f"{type}: mean_gini={m:.5f}, std={s:.5f}")


In [ ]:
X_train = X_train.drop(columns=["date", "day", "dayofweek", "month", "is_weekend", "dow_sin", "dow_cos", "month_sin", "month_cos"])
X_test  = X_test.drop(columns=["date", "day", "dayofweek", "month", "is_weekend", "dow_sin", "dow_cos", "month_sin", "month_cos"])


**Вывод.** Date features почти не улучшают качество: локально они дают около `0.002` Gini. В дальнейшем они не являются основной частью baseline.


### Результаты блока: date/region validation

| Набор признаков | Mean CV Gini | Std |
|---|---:|---:|
| Date features | 0.0023 | 0.0075 |
| Region target encoding | 0.0753 | 0.0033 |
| Date + region | 0.0711 | 0.0138 |

Интерпретация: даты сами по себе почти бесполезны, а region даёт небольшой устойчивый сигнал. Добавление date features к region не улучшило качество, поэтому в baseline я не делал на них ставку.


## 4. MMR как числовой сигнал

Средний MMR матча должен отражать уровень игры, но распределение признака скошено и содержит пропуски. Проверяю raw MMR, missing indicator и простые преобразования вроде `sqrt`.


In [ ]:
val_all_ind = np.concatenate([val_ind for _, val_ind in ts.split(X_train)])
val_all_ind = np.unique(val_all_ind)

X_val_all = X_train.iloc[val_all_ind].copy()

train_mmr = X_train[["avg_mmr"]].copy()
train_mmr["split"] = "train"

val_mmr = X_val_all[["avg_mmr"]].copy()
val_mmr["split"] = "validation_cv"

test_mmr = X_test[["avg_mmr"]].copy()
test_mmr["split"] = "test"

mmr = pd.concat([train_mmr, val_mmr, test_mmr], ignore_index=True)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.set_title("Распределение avg_mmr", fontsize=15)

sns.histplot(data=mmr, x="avg_mmr", hue="split", kde=True, bins=30, ax=ax)
plt.show()


In [ ]:
s = X_train["avg_mmr"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(np.log1p(s), bins=40, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("log(1+x)")

sns.histplot(np.sqrt(s), bins=40, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("sqrt(x)")

sns.histplot(1/(1+s), bins=40, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("1/(1+x)")

sns.histplot(np.exp(np.log(s[s > 0])), bins=40, kde=True, ax=axes[1, 1])
axes[1, 1].set_title("exp(log(x))")

plt.tight_layout()
plt.show()


In [ ]:
def count_mmr(ts, use_sqrt=False):
    train_gini, val_gini = [], []

    for tr_ind, val_ind in ts.split(X_train):
        X_tr = X_train.iloc[tr_ind].copy()
        X_val = X_train.iloc[val_ind].copy()
        y_tr = y_train.iloc[tr_ind]
        y_val = y_train.iloc[val_ind]

        enc = TargetEncoder(cols=["region"])
        X_tr["region_te"] = enc.fit_transform(X_tr[["region"]], y_tr)["region"]
        X_val["region_te"] = enc.transform(X_val[["region"]])["region"]

        mmr_median = X_tr["avg_mmr"].median()
        for df in (X_tr, X_val):
            df["mmr_missing"] = df["avg_mmr"].isna().astype(int)
            df["mmr_filled"] = df["avg_mmr"].fillna(mmr_median)
            df["mmr_feat"] = np.sqrt(df["mmr_filled"]) if use_sqrt else df["mmr_filled"]

        cols = ["region_te", "mmr_missing", "mmr_feat"]
        model = LogisticRegression(max_iter=1000)

        model.fit(X_tr[cols], y_tr)
        p_tr = model.predict_proba(X_tr[cols])[:, 1]
        p_val = model.predict_proba(X_val[cols])[:, 1]

        train_gini.append(gini(y_tr, p_tr))
        val_gini.append(gini(y_val, p_val))

    return np.mean(train_gini), np.std(train_gini), np.mean(val_gini), np.std(val_gini)

raw_tr, raw_tr_std, raw_val, raw_val_std = count_mmr(ts, use_sqrt=False)
sqrt_tr, sqrt_tr_std, sqrt_val, sqrt_val_std = count_mmr(ts, use_sqrt=True)

print(f"RAW  mmr: train_gini={raw_tr:.5f} ± {raw_tr_std:.5f}, val_gini={raw_val:.5f} ± {raw_val_std:.5f}")
print(f"SQRT mmr: train_gini={sqrt_tr:.5f} ± {sqrt_tr_std:.5f}, val_gini={sqrt_val:.5f} ± {sqrt_val_std:.5f}")


**Вывод.** MMR даёт стабильный умеренный сигнал: `Region + MMR` улучшает baseline примерно до `0.148` CV Gini. `sqrt(MMR)` оказался немного устойчивее raw MMR.


### Результаты блока: MMR

| Вариант MMR | Train Gini | Validation Gini |
|---|---:|---:|
| Raw MMR | 0.1458 ± 0.0003 | 0.1469 ± 0.0013 |
| `sqrt(MMR)` | 0.1466 ± 0.0003 | 0.1477 ± 0.0013 |

Интерпретация: MMR полезен, но не решает задачу сам по себе. `sqrt` немного стабилизирует распределение, а missing indicator важен, потому что пропуски MMR не случайны.


## 5. Драфт героев

Главная идея baseline: исход матча сильно зависит от выбранных героев. Поэтому я очищаю `player_df` до корректных 5v5 матчей и кодирую драфт как signed sparse vector: Radiant `+1`, Dire `-1`.


In [ ]:
player_df = pd.read_csv("dota-2-hse-ml-1-course-competition-2026/player_df.csv")
heroes_df = pd.read_csv("dota-2-hse-ml-1-course-competition-2026/Constants.Heroes.csv")

display(player_df.head())
display(heroes_df.head())
print(player_df.shape, heroes_df.shape)


In [ ]:
player_df["account_id"].value_counts().head(5)


In [ ]:
player_df = player_df[player_df["account_id"] != -1].copy()


In [ ]:
group_by_match_id = player_df.groupby("match_id")["hero_id"]

bad_match_ids = group_by_match_id.size().loc[lambda x: x != group_by_match_id.nunique()].index
print("bad matches:", len(bad_match_ids))


In [ ]:
player_df = player_df[~player_df["match_id"].isin(bad_match_ids)].copy()


In [ ]:
df = player_df.copy()
print("start: ", df.shape)

# только матчи из train/test
all_matches = pd.Index(train_df["match_id"]).union(test_df["match_id"])
df = df[df["match_id"].isin(all_matches)].copy()
print("1) ", df.shape)

# убираем пропуски
df = df.dropna(subset=["match_id", "account_id", "hero_id", "player_slot"]).copy()

# матчи с hero_id == 0 убираем
hero0_matches = df.loc[df["hero_id"].eq(0), "match_id"].unique()
df = df[~df["match_id"].isin(hero0_matches)].copy()
print("2) ", df.shape)

# типизация + игра только за одну сторону
df["side"] = np.where(
    df["player_slot"].between(0, 4), "radiant",
    np.where(df["player_slot"].between(128, 132), "dire", pd.NA)
)
df = df[df["side"].notna()].copy()

pairs_sides = (
    df[df["account_id"] != 4294967295]
    .groupby(["match_id", "account_id"])["side"]
    .nunique()
)
bad_pairs = pairs_sides[pairs_sides > 1].index

pair_index = pd.MultiIndex.from_frame(df[["match_id", "account_id"]])
df = df[~pair_index.isin(bad_pairs)].copy()
print("3) ", df.shape)

# оставляем только 5v5
cnt = df.groupby(["match_id", "side"]).size().unstack(fill_value=0)
good_matches = cnt.index[(cnt.get("radiant", 0) == 5) & (cnt.get("dire", 0) == 5)]
df = df[df["match_id"].isin(good_matches)].copy()
print("4) ", df.shape)

final_cnt = df.groupby(["match_id", "side"]).size().unstack(fill_value=0)
assert (final_cnt["radiant"] == 5).all()
assert (final_cnt["dire"] == 5).all()

player_df_clean = df.drop(columns=["side"]).copy()
print("final:", player_df_clean.shape)


In [ ]:
from scipy.sparse import csr_array


class HeroesEncoder:
    def __init__(
        self, match_col="match_id", hero_col="hero_id", slot_col="player_slot"
    ):
        self.match_col = match_col
        self.hero_col = hero_col
        self.slot_col = slot_col
        self.hero_to_col_ = None
        self.feature_names_ = None

    def fit(self, X, y=None):
        heroes = (
            pd.Series(X[self.hero_col].dropna().astype(int).unique())
            .sort_values()
            .tolist()
        )
        self.hero_to_col_ = {h: i for i, h in enumerate(heroes)}
        self.feature_names_ = [f"hero_{h}" for h in heroes]
        return self

    def transform(self, X, y=None):
        if self.hero_to_col_ is None:
            raise ValueError("Encoder is not fitted.")

        if y is None:
            match_ids = pd.Index(X[self.match_col].drop_duplicates())
        else:
            match_ids = pd.Index(y)

        match_to_row = {m: i for i, m in enumerate(match_ids)}

        work = X[[self.match_col, self.hero_col, self.slot_col]].copy()
        slots = work[self.slot_col].to_numpy()

        side = np.zeros(len(work), dtype=np.int8)
        side[(slots >= 0) & (slots <= 4)] = 1
        side[(slots >= 128) & (slots <= 132)] = -1

        rows = work[self.match_col].map(match_to_row).to_numpy()
        cols = work[self.hero_col].map(self.hero_to_col_).to_numpy()

        valid = (side != 0) & (~pd.isna(rows)) & (~pd.isna(cols))

        rows = rows[valid].astype(np.int32)
        cols = cols[valid].astype(np.int32)
        data = side[valid].astype(np.int8)

        mat = csr_array(
            (data, (rows, cols)),
            shape=(len(match_ids), len(self.hero_to_col_)),
            dtype=np.int8,
        )
        mat.sum_duplicates()
        mat.data = np.sign(mat.data).astype(np.int8)
        return mat

    def get_feature_names_out(self):
        if self.feature_names_ is None:
            raise ValueError("Encoder is not fitted.")
        return np.array(self.feature_names_, dtype=object)


In [ ]:
from scipy.sparse import csr_matrix, hstack

hero_encoder = HeroesEncoder().fit(player_df_clean)
player_train_all = player_df_clean[player_df_clean["match_id"].isin(X_train["match_id"])]
X_hero_all = hero_encoder.transform(player_train_all, y=X_train["match_id"])

def evaluate_hero_models(ts):
    all_tr_scores, all_val_scores = [], []
    hero_tr_scores, hero_val_scores = [], []

    for tr_ind, val_ind in ts.split(X_train):
        X_tr = X_train.iloc[tr_ind].copy()
        X_val = X_train.iloc[val_ind].copy()
        y_tr = y_train.iloc[tr_ind]
        y_val = y_train.iloc[val_ind]

        te = TargetEncoder(cols=["region"])
        X_tr["region_te"] = te.fit_transform(X_tr[["region"]], y_tr)["region"]
        X_val["region_te"] = te.transform(X_val[["region"]])["region"]

        mmr_med = X_tr["avg_mmr"].median()
        for d in (X_tr, X_val):
            d["mmr_missing"] = d["avg_mmr"].isna().astype(int)
            d["mmr_filled"] = d["avg_mmr"].fillna(mmr_med)
            d["mmr_feat"] = np.sqrt(d["mmr_filled"])

        dense_tr = csr_matrix(X_tr[["region_te", "mmr_missing", "mmr_feat"]].to_numpy())
        dense_val = csr_matrix(X_val[["region_te", "mmr_missing", "mmr_feat"]].to_numpy())

        X_hero_tr = X_hero_all[tr_ind]
        X_hero_val = X_hero_all[val_ind]

        X_all_tr = hstack([dense_tr, X_hero_tr], format="csr")
        X_all_val = hstack([dense_val, X_hero_val], format="csr")

        model_all = LogisticRegression(max_iter=2000)
        model_hero = LogisticRegression(max_iter=2000)

        model_all.fit(X_all_tr, y_tr)
        model_hero.fit(X_hero_tr, y_tr)

        p_all_tr = model_all.predict_proba(X_all_tr)[:, 1]
        p_all_val = model_all.predict_proba(X_all_val)[:, 1]
        p_hero_tr = model_hero.predict_proba(X_hero_tr)[:, 1]
        p_hero_val = model_hero.predict_proba(X_hero_val)[:, 1]

        all_tr_scores.append(gini(y_tr, p_all_tr))
        all_val_scores.append(gini(y_val, p_all_val))
        hero_tr_scores.append(gini(y_tr, p_hero_tr))
        hero_val_scores.append(gini(y_val, p_hero_val))

    print(f"ALL (region+mmr+heroes): train={np.mean(all_tr_scores):.5f} ± {np.std(all_tr_scores):.5f}, "
          f"val={np.mean(all_val_scores):.5f} ± {np.std(all_val_scores):.5f}")
    print(f"HEROES: train={np.mean(hero_tr_scores):.5f} ± {np.std(hero_tr_scores):.5f}, "
          f"val={np.mean(hero_val_scores):.5f} ± {np.std(hero_val_scores):.5f}")

evaluate_hero_models(ts)


**Вывод.** Драфт героев — самый сильный простой сигнал. Sparse hero encoding даёт около `0.274` CV Gini отдельно и около `0.308` вместе с region/MMR features.


### Результаты блока: hero draft

| Набор признаков | Train Gini | Validation Gini |
|---|---:|---:|
| Только heroes | 0.2788 ± 0.0019 | 0.2736 ± 0.0029 |
| Region + MMR + heroes | 0.3122 ± 0.0021 | 0.3078 ± 0.0020 |

Интерпретация: signed sparse encoding драфта героев даёт самый заметный прирост среди простых признаков. Даже без сложных interaction features модель уже начинает учитывать силу и баланс пиков.


## 6. Подбор гиперпараметров и submission

После выбора набора признаков подбираю параметры логистической регрессии через Optuna и обучаю финальную модель для генерации Kaggle-style submission.


In [ ]:
import optuna


def objective(trial):
    params = {
        "solver": trial.suggest_categorical("solver", ["liblinear", "lbfgs"]),
        "max_iter": trial.suggest_int("max_iter", 600, 2400, step=200),
        "C": trial.suggest_float("C", 1e-4, 30, log=True),
    }

    fold_scores = []

    for fold_id, (tr_ind, val_ind) in enumerate(ts.split(X_train), start=1):
        X_tr = X_train.iloc[tr_ind].copy()
        X_val = X_train.iloc[val_ind].copy()
        y_tr = y_train.iloc[tr_ind]
        y_val = y_train.iloc[val_ind]

        enc = TargetEncoder(cols=["region"])
        X_tr["region_te"] = enc.fit_transform(X_tr[["region"]], y_tr)["region"]
        X_val["region_te"] = enc.transform(X_val[["region"]])["region"]
        mmr_med = X_tr["avg_mmr"].median()
        for part in (X_tr, X_val):
            part["mmr_missing"] = part["avg_mmr"].isna().astype(int)
            part["mmr_filled"] = part["avg_mmr"].fillna(mmr_med)
            part["mmr_feat"] = np.sqrt(part["mmr_filled"])

        X_dense_tr = csr_matrix(
            X_tr[["region_te", "mmr_missing", "mmr_feat"]].to_numpy()
        )
        X_dense_val = csr_matrix(
            X_val[["region_te", "mmr_missing", "mmr_feat"]].to_numpy()
        )

        X_hero_tr = X_hero_all[tr_ind]
        X_hero_val = X_hero_all[val_ind]

        X_all_tr = hstack([X_dense_tr, X_hero_tr], format="csr")
        X_all_val = hstack([X_dense_val, X_hero_val], format="csr")

        model = LogisticRegression(**params, random_state=42)
        model.fit(X_all_tr, y_tr)
        p_val = model.predict_proba(X_all_val)[:, 1]

        score = gini(y_val, p_val)
        fold_scores.append(score)

        trial.report(score, step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)
study.optimize(objective, n_trials=25, show_progress_bar=True)


In [ ]:
print("best_gini:", study.best_value)
print("best_params:", study.best_params)


In [ ]:
from optuna.visualization import plot_param_importances, plot_optimization_history

fig_imp = plot_param_importances(study)
fig_imp.show()

fig_hist = plot_optimization_history(study)
fig_hist.show()


In [ ]:
best_params = study.best_params
best_cv_gini = float(study.best_value)

print("best_params:", best_params)
print("best_cv_gini:", best_cv_gini)

hero_encoder = HeroesEncoder().fit(player_df_clean)

player_train_all = player_df_clean[player_df_clean["match_id"].isin(X_train["match_id"])]
player_test_all = player_df_clean[player_df_clean["match_id"].isin(X_test["match_id"])]

X_hero_train = hero_encoder.transform(player_train_all, y=X_train["match_id"])
X_hero_test = hero_encoder.transform(player_test_all, y=X_test["match_id"])

te_final = TargetEncoder(cols=["region"])
region_train = te_final.fit_transform(X_train[["region"]], y_train)["region"]
region_test = te_final.transform(X_test[["region"]])["region"]

mmr_med = X_train["avg_mmr"].median()
mmr_missing_train = X_train["avg_mmr"].isna().astype(int)
mmr_missing_test = X_test["avg_mmr"].isna().astype(int)
mmr_feat_train = np.sqrt(X_train["avg_mmr"].fillna(mmr_med))
mmr_feat_test = np.sqrt(X_test["avg_mmr"].fillna(mmr_med))

X_region_mmr_train = csr_matrix(np.c_[region_train, mmr_missing_train, mmr_feat_train])
X_region_mmr_test = csr_matrix(np.c_[region_test, mmr_missing_test, mmr_feat_test])

X_all_train = hstack([X_region_mmr_train, X_hero_train], format="csr")
X_all_test = hstack([X_region_mmr_test, X_hero_test], format="csr")

model_final = LogisticRegression(**best_params, random_state=42)
model_final.fit(X_all_train, y_train)

test_pred = model_final.predict_proba(X_all_test)[:, 1]
submission = pd.DataFrame({
    "ID": X_test["match_id"].values,
    "Value": test_pred
})
submission.to_csv("submission_all_features_base.csv", index=False)

print("saved")


## Итоги baseline

- Date features почти не дают самостоятельного сигнала.
- Region target encoding полезен, но требует аккуратной валидации.
- MMR добавляет стабильный умеренный вклад, особенно вместе с missing indicator.
- Самый сильный прирост даёт sparse-кодирование драфта героев.
- Лучший сохранённый Optuna CV Gini для packaged baseline: `0.4089`.
- Итоговый Kaggle score: `0.38787`, место `42 / 444`.


### Финальные результаты

| Метрика | Значение |
|---|---:|
| Лучший сохранённый CV Gini | 0.4089 |
| Kaggle score | 0.38787 |
| Место на leaderboard | 42 / 444 |

Главный итог: для этой постановки сильный baseline строится не вокруг сложной модели, а вокруг аккуратного feature engineering: корректная очистка `player_df`, sparse-драфт героев, region/MMR preprocessing и честная валидация.
